# RFP Parsing P2 250 Pipeline

Goal: build V1/V2 수정버전 outputs for a 250-document pilot set.

- This is not V3.
- Output folder: `outputs/parsing_p2_250`
- Existing `outputs/parsing_v1_v2_2차` is not overwritten.
- V1/V2 chunking is preserved: 1000 chars / 150 overlap.
- Added P2 fixes: unique chunk_id, toc chunk separation, artifact cleaner reinforcement.


## Large File Note

원본 HWP/PDF와 parsing output(JSONL/XLSX/CSV)은 용량이 크기 때문에 GitHub에 포함하지 않습니다. 실행 전 공유 Drive 또는 로컬/VM 디스크에 `data/`와 `outputs/` 구조를 먼저 배치합니다.


In [1]:
# 0. Install requirements for a fresh virtual environment
# This cell is self-contained. Run it once in a new environment, then restart the kernel.
# For automated re-execution in an already prepared environment, set RFP_SKIP_INSTALL=1.

from pathlib import Path
import os
import subprocess
import sys


def find_project_dir_for_install() -> Path:
    candidates = []
    env_project_dir = os.getenv("RFP_PROJECT_DIR")
    if env_project_dir:
        candidates.append(Path(env_project_dir).expanduser().resolve())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    seen = set()
    unique_candidates = []
    for path in candidates:
        if path not in seen:
            seen.add(path)
            unique_candidates.append(path)

    for path in unique_candidates:
        if (path / "requirements.txt").exists():
            return path

    checked = "\n".join(str(path) for path in unique_candidates[:12])
    raise FileNotFoundError(
        "requirements.txt not found. Run this notebook inside project_2nd, "
        "or set RFP_PROJECT_DIR to the project_2nd path.\n\n"
        f"checked:\n{checked}"
    )


INSTALL_PROJECT_DIR = find_project_dir_for_install()
INSTALL_REQUIREMENTS_TXT = INSTALL_PROJECT_DIR / "requirements.txt"

print("PROJECT_DIR:", INSTALL_PROJECT_DIR)
if os.getenv("RFP_SKIP_INSTALL") == "1":
    print("RFP_SKIP_INSTALL=1, requirements installation skipped for this run.")
else:
    cmd = [sys.executable, "-m", "pip", "install", "-r", str(INSTALL_REQUIREMENTS_TXT)]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
    print("requirements installed. Restart the kernel, then continue from the next cell.")


PROJECT_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd
RFP_SKIP_INSTALL=1, requirements installation skipped for this run.


In [2]:
# 1. Project path configuration
# Large input files are intentionally not committed. Place data/outputs on Drive, local disk, or VM before running.
# Teammates should only need to adjust the working directory or RFP_PROJECT_DIR.

from pathlib import Path
import os
import sys


def find_project_dir() -> Path:
    candidates = []
    env_project_dir = os.getenv("RFP_PROJECT_DIR")
    if env_project_dir:
        candidates.append(Path(env_project_dir).expanduser().resolve())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    seen = set()
    unique_candidates = []
    for path in candidates:
        if path not in seen:
            seen.add(path)
            unique_candidates.append(path)

    for path in unique_candidates:
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path

    checked = "\n".join(str(path) for path in unique_candidates[:12])
    raise FileNotFoundError(
        "Project root not found. Run this notebook inside project_2nd, "
        "or set RFP_PROJECT_DIR to the project_2nd path.\n\n"
        f"checked:\n{checked}"
    )


PROJECT_DIR = find_project_dir()
PARSING_OUTPUT_NAME = "parsing_p2_250"
PARSING_VERSION_LABEL = "p2_chunkfix_toc_clean"

PARSING_SRC_DIR = PROJECT_DIR / "src" / "parsing"
for import_path in [PARSING_SRC_DIR, PROJECT_DIR]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from rfp_parsing_v1_v2_lib import make_project_paths

PATHS = make_project_paths(
    PROJECT_DIR,
    parsing_output_name=PARSING_OUTPUT_NAME,
    parsing_version_label=PARSING_VERSION_LABEL,
)

print("PROJECT_DIR:", PROJECT_DIR)
print("PARSING_OUTPUT_NAME:", PARSING_OUTPUT_NAME)
print("PARSING_VERSION_LABEL:", PARSING_VERSION_LABEL)
print("OUTPUT_DESCRIPTION:", PATHS["output_description"])
print("DATA_DIR:", PATHS["data_dir"])
print("ORIGINAL_DATA_DIR:", PATHS["original_data_dir"])
print("EVAL_DIR:", PATHS["eval_dir"])
print("PARSING_OUTPUT_DIR:", PATHS["parsing_output_dir"])
print("METADATA_EXCEL:", PATHS["metadata_excel"])
print("Python:", sys.executable)


PROJECT_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd
PARSING_OUTPUT_NAME: parsing_p2_250
PARSING_VERSION_LABEL: p2_chunkfix_toc_clean
OUTPUT_DESCRIPTION: P2 - chunk_id 중복 수정, toc 분리, artifact cleaner 보강
DATA_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd\data
ORIGINAL_DATA_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd\data\original_data_list
EVAL_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd\data\eval
PARSING_OUTPUT_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250
METADATA_EXCEL: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\rfp_parsing_metadata_250_p2_chunkfix_toc_clean.xlsx
Python: C:\Users\yoosy\Desktop\codeit\project_2nd\.venv\Scripts\python.exe


C:\Users\yoosy\Desktop\codeit\project_2nd\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Fixed Scope

- Pilot: 250 documents total, including all 62 eval ground-truth documents.
- V1: section/chapter-aware text blocks.
- V2: V1 text blocks plus table-like row/dict blocks and budget/submission candidates.
- Deferred: image OCR, vision embeddings, exact page matching.
- Required summary: document counts, success/failure counts, block counts, and chunk counts.


In [3]:
# 2. Load constants and artifact policy

from rfp_parsing_v1_v2_lib import (
    PILOT_TOTAL_DOCS,
    EVAL_GROUND_TRUTH_DOCS_TOTAL,
    EVAL_PHYSICAL_SOURCE_DOCS_TOTAL,
    ADDITIONAL_SAMPLED_DOCS,
    ARTIFACT_REMOVE_TOKENS,
    CONFIRMED_KEEP_HANJA_TOKENS,
    KEEP_HANJA_RUNS,
    CHUNK_MAX_CHARS,
    CHUNK_OVERLAP,
)

print("pilot docs:", PILOT_TOTAL_DOCS)
print("eval ground-truth aliases:", EVAL_GROUND_TRUTH_DOCS_TOTAL)
print("eval physical source docs:", EVAL_PHYSICAL_SOURCE_DOCS_TOTAL)
print("additional sampled docs:", ADDITIONAL_SAMPLED_DOCS)
print("artifact remove tokens:", sorted(ARTIFACT_REMOVE_TOKENS))
print("confirmed keep hanja tokens:", sorted(CONFIRMED_KEEP_HANJA_TOKENS))
print("keep hanja runs:", sorted(KEEP_HANJA_RUNS))
print("chunk max chars:", CHUNK_MAX_CHARS)
print("chunk overlap:", CHUNK_OVERLAP)
print("retrieval signals: project_name, issuer, final_notice_id, final_budget, final dates, final periods/deadlines, final eligibility terms, exact_terms, dates, amounts, submission_doc_terms")


pilot docs: 250
eval ground-truth aliases: 62
eval physical source docs: 61
additional sampled docs: 189
artifact remove tokens: ['普', '楴', '汫', '沤', '浥', '浫', '浵', '潣', '爔', '牦', '蕀', '遠']
confirmed keep hanja tokens: ['丁', '丙', '乙', '內', '共', '外', '新', '有', '未', '案', '無', '甲', '舊', '過']
keep hanja runs: ['內外', '共有', '有償', '未定', '案內', '無償', '甲乙', '甲乙丙', '甲乙丙丁']
chunk max chars: 1000
chunk overlap: 150
retrieval signals: project_name, issuer, final_notice_id, final_budget, final dates, final periods/deadlines, final eligibility terms, exact_terms, dates, amounts, submission_doc_terms


In [4]:
# 3. Build pilot_docs_250.csv
# Includes the 62 eval documents first, then fills the remaining 188 with budget/submission/evaluation/form-heavy documents.

from rfp_parsing_v1_v2_lib import build_pilot_docs

pilot_docs_df, eval_docs_df, original_inventory_df, metadata_df = build_pilot_docs(PATHS)

print("saved:", PATHS["pilot_docs_csv"])
print("eval ground truth docs:", len(eval_docs_df))
print("original inventory docs:", len(original_inventory_df))
print("metadata rows:", len(metadata_df))
print("pilot_total_docs:", len(pilot_docs_df))
print("eval_aliases_covered:", EVAL_GROUND_TRUTH_DOCS_TOTAL)
print("eval_physical_source_docs_included:", int(pilot_docs_df["is_eval_ground_truth"].sum()))
print("additional_sampled_docs:", int((~pilot_docs_df["is_eval_ground_truth"].astype(bool)).sum()))
print("\nfile_type counts:")
print(pilot_docs_df["file_type"].value_counts().to_string())
print("\nsampling_reason counts:")
print(pilot_docs_df["sampling_reason"].value_counts().head(20).to_string())

display(pilot_docs_df.head(10))


saved: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\pilot_docs_250.csv
eval ground truth docs: 61
original inventory docs: 690
metadata rows: 0
pilot_total_docs: 250
eval_aliases_covered: 62
eval_physical_source_docs_included: 61
additional_sampled_docs: 189

file_type counts:
file_type
hwp    247
pdf      3

sampling_reason counts:
sampling_reason
appendix_forms+budget+eligibility+evaluation_table+submission_docs                 126
eval_ground_truth                                                                   61
appendix_forms+eligibility+evaluation_table+submission_docs                         26
appendix_forms+budget+eligibility+submission_docs                                   23
appendix_forms+budget+evaluation_table+submission_docs                               7
appendix_forms+eligibility+submission_docs                                           2
budget+eligibility+evaluation_table+submission_docs                                  2
appendix_forms+budge

,pilot_index,norm_name,doc_id,source_file,source_path,file_type,is_eval_ground_truth,sampling_reason,sample_score,eval_question_count,...,notice_id,notice_round,published_at,bid_start,bid_deadline,pilot_doc_id,external_notice_id,final_notice_id,notice_id_status,notice_id_evidence
0,1,고려대학교_차세대 포털·학사 정보시스템 구축사업,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,C:\Users\yoosy\Desktop\codeit\project_2nd\data...,pdf,True,eval_ground_truth,1,42,...,,,,,,P001,,,missing,
1,2,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시,doc_d9e0aa81c1,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원...,C:\Users\yoosy\Desktop\codeit\project_2nd\data...,hwp,True,eval_ground_truth,46,30,...,,,,,,P002,,,missing,
2,3,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스,doc_3f6e8b8c36,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp,C:\Users\yoosy\Desktop\codeit\project_2nd\data...,hwp,True,eval_ground_truth,126,27,...,,,,,,P003,,,missing,
3,4,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능,doc_5ec6caabff,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp,C:\Users\yoosy\Desktop\codeit\project_2nd\data...,hwp,True,eval_ground_truth,56,26,...,,,,,,P004,,,missing,
4,5,한국원자력연구원_한국원자력연구원 선량평가시스템 고도화,doc_46598b539b,한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp,C:\Users\yoosy\Desktop\codeit\project_2nd\data...,hwp,True,eval_ground_truth,35,26,...,,,,,,P005,,,missing,
5,6,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역,doc_de2e55f147,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,C:\Users\yoosy\Desktop\codeit\project_2nd\data...,hwp,True,eval_ground_truth,37,26,...,,,,,,P006,,,missing,
6,7,울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역,doc_6886cf0cd5,울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp,C:\Users\yoosy\Desktop\codeit\project_2nd\data...,hwp,True,eval_ground_truth,83,25,...,,,,,,P007,,,missing,
7,8,케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템,doc_e317bfe13b,케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp,C:\Users\yoosy\Desktop\codeit\project_2nd\data...,hwp,True,eval_ground_truth,73,25,...,,,,,,P008,,,missing,
8,9,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축,doc_3a265f371a,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,C:\Users\yoosy\Desktop\codeit\project_2nd\data...,hwp,True,eval_ground_truth,9,25,...,,,,,,P009,,,missing,
9,10,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원,doc_fd59006ed9,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,C:\Users\yoosy\Desktop\codeit\project_2nd\data...,hwp,True,eval_ground_truth,64,23,...,,,,,,P010,,,missing,


In [5]:
# 4. Run V1/V2 parsing pipeline
# Reads the 250 pilot documents and writes parsed_blocks/chunks/summary outputs.

from rfp_parsing_v1_v2_lib import run_parsing_pipeline, create_metadata_excel

summary, doc_parse_summary_df = run_parsing_pipeline(PATHS, pilot_docs_df=pilot_docs_df)
metadata_excel_path = create_metadata_excel(PATHS)

print("saved outputs:")
for key in [
    "pilot_docs_csv",
    "doc_parse_summary_csv",
    "parsed_blocks_v1_jsonl",
    "parsed_blocks_v2_jsonl",
    "chunks_v1_jsonl",
    "chunks_v2_jsonl",
    "parsing_summary_json",
    "parsing_summary_md",
    "metadata_excel",
]:
    print("-", PATHS[key])

print("\nsummary:")
for key, value in summary.items():
    if isinstance(value, list):
        print(f"{key}: {len(value)} items")
    else:
        print(f"{key}: {value}")

display(doc_parse_summary_df.head(10))


parse pilot docs:   0%|          | 0/250 [00:00<?, ?it/s]

parse pilot docs:   0%|          | 1/250 [00:17<1:13:00, 17.59s/it]

parse pilot docs:   1%|          | 2/250 [00:18<30:56,  7.49s/it]  

parse pilot docs:   1%|          | 3/250 [00:18<17:28,  4.24s/it]

parse pilot docs:   2%|▏         | 4/250 [00:19<11:31,  2.81s/it]

...

parse pilot docs: 100%|█████████▉| 249/250 [02:48<00:00,  2.23it/s]

parse pilot docs: 100%|██████████| 250/250 [02:49<00:00,  2.23it/s]

parse pilot docs: 100%|██████████| 250/250 [02:49<00:00,  1.48it/s]

saved outputs:
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\pilot_docs_250.csv
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\doc_parse_summary.csv
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\parsed_blocks_v1.jsonl
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\parsed_blocks_v2.jsonl
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\chunks_v1.jsonl
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\chunks_v2.jsonl
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\parsing_summary.json
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\parsing_summary.md
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\rfp_parsing_metadata_250_p2_chunkfix_toc_clean.xlsx

summary:
output_description: P2 - chunk_id 중복 수정, toc 분리, artifact cleaner 보강
parsing_output_name: parsing_p2_250
parsing_version_label: p2_chunkfix_toc_clean
pilot_total

,pilot_doc_id,doc_id,norm_name,source_file,file_type,is_eval_ground_truth,parser_status,parser,raw_char_len,clean_char_len,...,final_budget_krw,final_budget_status,final_published_at,final_bid_start,final_bid_deadline,final_project_duration,final_maintenance_period,final_warranty_period,final_deadline_terms,final_bid_eligibility_terms
0,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,pdf,True,success,pypdf_extract_text,229996,188946,...,11270000000,extracted,,,,계약일로부터 24개월 이내,사업종료일로부터 12개월,12개월,,- 공동수급업체 참여자는 입찰참가자격을 모두 충족하여야 함 | 1) 소프트웨어산업진...
1,P002,doc_d9e0aa81c1,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원...,hwp,True,success,olefile_hwp_bodytext,56731,50340,...,243000000,extracted,,,2024-06-11,15일,,1년,,다. 국가종합전자조달시스템 입찰참가자격등록규정에 의하여 나라장터(G2B시스템)에 입...
2,P003,doc_3f6e8b8c36,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp,hwp,True,success,olefile_hwp_bodytext,53721,46801,...,,candidate_only,,,,계약체결일로부터 2027년 12월 31일 까지,사업종료후 1개월 이내,,별도 통보,"1. 입찰참가자격 | ❍ 참여업체간 담합, 평가위원 및 사무국 관계직원에 대한 뇌물..."
3,P004,doc_5ec6caabff,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp,hwp,True,success,olefile_hwp_bodytext,76189,69081,...,5000000000,extracted,,,,계약체결일로부터 120일,사업종료 후 1년 이내,사업종료 후 1년 이내,공고문 참조,② 「중소기업자간 경쟁제품 직접생산확인기준」에 의거 직접생산증명서 [입찰마감일 전일...
4,P005,doc_46598b539b,한국원자력연구원_한국원자력연구원 선량평가시스템 고도화,한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp,hwp,True,success,olefile_hwp_bodytext,47605,42445,...,46600000,extracted,,,,계약일로부터 6개월,1년,완료일로부터 12개월,,○ 「국가를 당사자로 하는 계약에 관한 법률 시행령 제12조(경쟁입찰의 참가자격) ...
5,P006,doc_de2e55f147,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,hwp,True,success,olefile_hwp_bodytext,79604,69798,...,1500000000,extracted,,,,5개월 이내,1년,1년,공고문 참조,"3. 공동수급체 구성원중 파산, 해산, 부도 기타 정당한 이유없이 해당 계약을 이행..."
6,P007,doc_6886cf0cd5,울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역,울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp,hwp,True,success,olefile_hwp_bodytext,120250,108194,...,986945000,extracted,,,,착수일로부터 5개월,,1년,,1) 정보통신공사업법 제3조 및 14조 규정에 따라 정보통신공사업(업종코드 0036...
7,P008,doc_e317bfe13b,케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템,케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp,hwp,True,success,olefile_hwp_bodytext,53737,45521,...,310000000,extracted,,,,계약체결일로부터 2025년 12월 31일 까지,3년간,,공고문 참조,- 「국가종합전자조달시스템 입찰참가자격등록 규정(조달청)」에 의하여 반드시 입찰일 ...
8,P009,doc_3a265f371a,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,hwp,True,success,olefile_hwp_bodytext,107813,99359,...,,missing,,,,계약일로부터 28개월,,,,사. 하도급 제한 규정 위반시 시정요구 | ○ 계약상대자가 「소프트웨어 진흥법」제5...
9,P010,doc_fd59006ed9,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,hwp,True,success,olefile_hwp_bodytext,141784,127016,...,8000000000,extracted,,,,계약체결일로부터 2027년 9월 30일까지,,1년간,공고문 참조,- 소프트웨어사업자(컴퓨터관련서비스사업) (나라장터 업종코드 : 1468) | - ...


In [6]:
# 5. Final validation and quick preview

import json
import pandas as pd

summary = json.loads(PATHS["parsing_summary_json"].read_text(encoding="utf-8"))

assert summary["pilot_total_docs"] == PILOT_TOTAL_DOCS
assert summary["eval_docs_included"] == EVAL_GROUND_TRUTH_DOCS_TOTAL
assert summary["eval_physical_source_docs_included"] == EVAL_PHYSICAL_SOURCE_DOCS_TOTAL
assert summary["parse_success_docs"] + summary["parse_failed_docs"] == PILOT_TOTAL_DOCS
assert summary["v2_chunks"] >= summary["v1_chunks"] or summary["v2_chunks"] == 0
assert summary["parsing_output_name"] == PARSING_OUTPUT_NAME
assert summary["parsing_version_label"] == PARSING_VERSION_LABEL

unknown_notice_count = 0
with PATHS["chunks_v2_jsonl"].open("r", encoding="utf-8") as f:
    for line in f:
        if "공고번호: unknown" in line:
            unknown_notice_count += 1
assert unknown_notice_count == 0

print("validation passed")
print("summary path:", PATHS["parsing_summary_json"])
print("markdown summary path:", PATHS["parsing_summary_md"])
print("metadata excel path:", PATHS["metadata_excel"])
print("unknown notice header count:", unknown_notice_count)
for key in [
    "docs_with_final_budget",
    "docs_with_date_candidates",
    "docs_with_final_bid_deadline",
    "docs_with_period_candidates",
    "docs_with_final_project_duration",
    "docs_with_final_maintenance_period",
    "docs_with_final_warranty_period",
    "docs_with_final_deadline_terms",
    "docs_with_eligibility_candidates",
    "docs_with_final_bid_eligibility_terms",
]:
    print(f"{key}:", summary.get(key))

def show_df(title: str, df: pd.DataFrame):
    print(f"\n### {title}")
    display(df)

preview_doc_summary = pd.read_csv(PATHS["doc_parse_summary_csv"], encoding="utf-8-sig")
show_df("\ubb38\uc11c \ud30c\uc2f1 \uc694\uc57d df - \uc0c1\uc704 10\ud589", preview_doc_summary.head(10))

metadata_preview = pd.read_excel(PATHS["metadata_excel"], sheet_name="metadata_250")
print("\nmetadata columns:")
print(metadata_preview.columns.tolist())
show_df("metadata key columns df - first 5 rows", metadata_preview[[
    "\uacf5\uace0 \ubc88\ud638",
    "\uc0ac\uc5c5\uba85",
    "\uc0ac\uc5c5 \uae08\uc561",
    "\uc0ac\uc5c5\uae08\uc561_\uc6d0",
    "\uc0ac\uc5c5\uae30\uac04",
    "\ubb34\uc0c1\uc720\uc9c0\ubcf4\uc218\uae30\uac04",
    "\ud558\uc790\ub2f4\ubcf4\ucc45\uc784\uae30\uac04",
    "\uae30\ud55c/\uae30\uac04 \uae30\ud0c0",
    "\uc785\ucc30\ucc38\uac00\uc790\uaca9",
    "\uc81c\ucd9c\uc11c\ub958",
    "\uc81c\ucd9c\uc11c\ub958_\uadfc\uac70\ubb36\uc74c",
    "\uc6d0\ubb38\ud14d\uc2a4\ud2b8_20000\uc790",
]].head(5))

period_candidates_preview = pd.read_excel(PATHS["metadata_excel"], sheet_name="period_candidates")
eligibility_candidates_preview = pd.read_excel(PATHS["metadata_excel"], sheet_name="eligibility_candidates")
print("\nperiod candidate rows:", len(period_candidates_preview))
print("eligibility candidate rows:", len(eligibility_candidates_preview))
show_df("\uae30\uac04/\uae30\ud55c \ud6c4\ubcf4 df - \uc0c1\uc704 10\ud589", period_candidates_preview.head(10))
show_df("\uc785\ucc30\ucc38\uac00\uc790\uaca9 \ud6c4\ubcf4 df - \uc0c1\uc704 10\ud589", eligibility_candidates_preview.head(10))

with PATHS["chunks_v2_jsonl"].open("r", encoding="utf-8") as f:
    for _ in range(3):
        line = f.readline().strip()
        if not line:
            break
        item = json.loads(line)
        print("\n--- chunk preview ---")
        print("chunk_id:", item["chunk_id"])
        print("source_file:", item["source_file"])
        print("chunk_type:", item["chunk_type"])
        print("issuer:", item.get("issuer"))
        print("project_name:", item.get("project_name"))
        print("final_notice_id:", item.get("final_notice_id"))
        print("final_budget:", item.get("final_budget"))
        print("final_bid_deadline:", item.get("final_bid_deadline"))
        print("final_project_duration:", item.get("final_project_duration"))
        print("final_maintenance_period:", item.get("final_maintenance_period"))
        print("final_warranty_period:", item.get("final_warranty_period"))
        print("final_deadline_terms:", item.get("final_deadline_terms"))
        print("final_bid_eligibility_terms:", str(item.get("final_bid_eligibility_terms", ""))[:200])
        print("exact_terms:", item.get("exact_terms", [])[:8])
        print("amounts:", item.get("amounts", [])[:5])
        print("dates:", item.get("dates", [])[:5])
        print(item["content"][:500])


validation passed
summary path: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\parsing_summary.json
markdown summary path: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\parsing_summary.md
metadata excel path: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\rfp_parsing_metadata_250_p2_chunkfix_toc_clean.xlsx
unknown notice header count: 0
docs_with_final_budget: 238
docs_with_date_candidates: 161
docs_with_final_bid_deadline: 40
docs_with_period_candidates: 250
docs_with_final_project_duration: 245
docs_with_final_maintenance_period: 53
docs_with_final_warranty_period: 203
docs_with_final_deadline_terms: 151
docs_with_eligibility_candidates: 250
docs_with_final_bid_eligibility_terms: 250

### 문서 파싱 요약 df - 상위 10행


,pilot_doc_id,doc_id,norm_name,source_file,file_type,is_eval_ground_truth,parser_status,parser,raw_char_len,clean_char_len,...,final_budget_krw,final_budget_status,final_published_at,final_bid_start,final_bid_deadline,final_project_duration,final_maintenance_period,final_warranty_period,final_deadline_terms,final_bid_eligibility_terms
0,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,pdf,True,success,pypdf_extract_text,229996,188946,...,1.127000e+10,extracted,NaN,NaN,NaN,계약일로부터 24개월 이내,사업종료일로부터 12개월,12개월,NaN,- 공동수급업체 참여자는 입찰참가자격을 모두 충족하여야 함 | 1) 소프트웨어산업진...
1,P002,doc_d9e0aa81c1,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원...,hwp,True,success,olefile_hwp_bodytext,56731,50340,...,2.430000e+08,extracted,NaN,NaN,2024-06-11,15일,NaN,1년,NaN,다. 국가종합전자조달시스템 입찰참가자격등록규정에 의하여 나라장터(G2B시스템)에 입...
2,P003,doc_3f6e8b8c36,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp,hwp,True,success,olefile_hwp_bodytext,53721,46801,...,NaN,candidate_only,NaN,NaN,NaN,계약체결일로부터 2027년 12월 31일 까지,사업종료후 1개월 이내,NaN,별도 통보,"1. 입찰참가자격 | ❍ 참여업체간 담합, 평가위원 및 사무국 관계직원에 대한 뇌물..."
3,P004,doc_5ec6caabff,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp,hwp,True,success,olefile_hwp_bodytext,76189,69081,...,5.000000e+09,extracted,NaN,NaN,NaN,계약체결일로부터 120일,사업종료 후 1년 이내,사업종료 후 1년 이내,공고문 참조,② 「중소기업자간 경쟁제품 직접생산확인기준」에 의거 직접생산증명서 [입찰마감일 전일...
4,P005,doc_46598b539b,한국원자력연구원_한국원자력연구원 선량평가시스템 고도화,한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp,hwp,True,success,olefile_hwp_bodytext,47605,42445,...,4.660000e+07,extracted,NaN,NaN,NaN,계약일로부터 6개월,1년,완료일로부터 12개월,NaN,○ 「국가를 당사자로 하는 계약에 관한 법률 시행령 제12조(경쟁입찰의 참가자격) ...
5,P006,doc_de2e55f147,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,hwp,True,success,olefile_hwp_bodytext,79604,69798,...,1.500000e+09,extracted,NaN,NaN,NaN,5개월 이내,1년,1년,공고문 참조,"3. 공동수급체 구성원중 파산, 해산, 부도 기타 정당한 이유없이 해당 계약을 이행..."
6,P007,doc_6886cf0cd5,울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역,울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp,hwp,True,success,olefile_hwp_bodytext,120250,108194,...,9.869450e+08,extracted,NaN,NaN,NaN,착수일로부터 5개월,NaN,1년,NaN,1) 정보통신공사업법 제3조 및 14조 규정에 따라 정보통신공사업(업종코드 0036...
7,P008,doc_e317bfe13b,케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템,케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp,hwp,True,success,olefile_hwp_bodytext,53737,45521,...,3.100000e+08,extracted,NaN,NaN,NaN,계약체결일로부터 2025년 12월 31일 까지,3년간,NaN,공고문 참조,- 「국가종합전자조달시스템 입찰참가자격등록 규정(조달청)」에 의하여 반드시 입찰일 ...
8,P009,doc_3a265f371a,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,hwp,True,success,olefile_hwp_bodytext,107813,99359,...,NaN,missing,NaN,NaN,NaN,계약일로부터 28개월,NaN,NaN,NaN,사. 하도급 제한 규정 위반시 시정요구 | ○ 계약상대자가 「소프트웨어 진흥법」제5...
9,P010,doc_fd59006ed9,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,hwp,True,success,olefile_hwp_bodytext,141784,127016,...,8.000000e+09,extracted,NaN,NaN,NaN,계약체결일로부터 2027년 9월 30일까지,NaN,1년간,공고문 참조,- 소프트웨어사업자(컴퓨터관련서비스사업) (나라장터 업종코드 : 1468) | - ...



metadata columns:
['공고 번호', '사업명', '사업 금액', '사업금액_원', '사업금액_상태', '사업금액_근거', '사업기간', '사업기간_근거', '무상유지보수기간', '무상유지보수기간_근거', '하자담보책임기간', '하자담보책임기간_근거', '기한/기간 기타', '기한/기간 기타_근거', '입찰참가자격', '입찰참가자격_근거', '발주 기관', '공개 일자', '입찰 참여 시작일', '입찰 참여 마감일', '공개일자_근거', '입찰시작일_근거', '입찰마감일_근거', '제출서류', '제출서류_근거묶음', '사업 요약', '원문텍스트_20000자', '파일형식', '파일명', 'pilot_doc_id', 'doc_id', 'is_eval_ground_truth', 'sampling_reason', 'parser_status', 'raw_char_len', 'clean_char_len', 'v1_text_blocks', 'v2_table_blocks', 'v1_chunks', 'v2_chunks', 'budget_candidate_count', 'date_candidate_count', 'period_candidate_count', 'eligibility_candidate_count', 'submission_doc_candidate_count', 'final_submission_doc_count', 'final_submission_document_name_count', 'source_path']

### metadata key columns df - first 5 rows


,공고 번호,사업명,사업 금액,사업금액_원,사업기간,무상유지보수기간,하자담보책임기간,기한/기간 기타,입찰참가자격,제출서류,제출서류_근거묶음,원문텍스트_20000자
0,NaN,차세대 포털·학사 정보시스템 구축사업,"11,270,000,000원",1.127000e+10,계약일로부터 24개월 이내,사업종료일로부터 12개월,12개월,NaN,- 공동수급업체 참여자는 입찰참가자격을 모두 충족하여야 함 | 1) 소프트웨어산업진...,"제안서, 발표자료, 사업자등록증","제안서, 발표자료 | 제안서 | 사업자등록증",2024. 7. 01\n목 차\nⅠ. 사업개요\n1. 사업 개요\n2. 사업 배경\...
1,NaN,2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시,"243,000,000원",2.430000e+08,15일,NaN,1년,NaN,다. 국가종합전자조달시스템 입찰참가자격등록규정에 의하여 나라장터(G2B시스템)에 입...,"제안서, 실적증명서, 확인서, 소프트웨어사업자, 가격제안서","제안서 | 실적증명서 | 확인서, 소프트웨어사업자 | 제안서, 가격제안서",목차\n목 차\n\n제 안 요 청 서\nBIFF&ACFM 온라인서비스 재개발 및\n...
2,NaN,우즈벡-키르기즈스탄 기후변화대응 스,NaN,NaN,계약체결일로부터 2027년 12월 31일 까지,사업종료후 1개월 이내,NaN,별도 통보,"1. 입찰참가자격 | ❍ 참여업체간 담합, 평가위원 및 사무국 관계직원에 대한 뇌물...","입찰서, 제안서, 제안요약서, 발표자료, 공동수급협정서, 가격제안서","입찰서 | 제안서, 제안요약서, 발표자료 | 제안서, 공동수급협정서 | 제안서, 발...",목 차\nⅠ. 제안 요청내용 01\n1. 용역개요 02\n2. 기본방향 03\n3....
3,NaN,의료기기산업 종합정보시스템(정보관리기관) 기능,50억원,5.000000e+09,계약체결일로부터 120일,사업종료 후 1년 이내,사업종료 후 1년 이내,공고문 참조,② 「중소기업자간 경쟁제품 직접생산확인기준」에 의거 직접생산증명서 [입찰마감일 전일...,"제안서, 발표자료, 보안서약서, 서약서, 확약서, 청렴계약","제안서, 발표자료 | 보안서약서, 서약서, 확약서 | 청렴계약, 서약서 | 보안서약...",목 차\n1. 사업개요 4\n2. 추진배경 및 필요성 4\n3. 과업내용 5\n4....
4,NaN,한국원자력연구원 선량평가시스템 고도화,"46,600 천원",4.660000e+07,계약일로부터 6개월,1년,완료일로부터 12개월,NaN,○ 「국가를 당사자로 하는 계약에 관한 법률 시행령 제12조(경쟁입찰의 참가자격) ...,"청렴계약, 보안서약서, 서약서, 확인서, 확약서","청렴계약, 보안서약서, 서약서, 확인서 | 보안서약서, 서약서, 확약서 | 보안서약...",1\n사업 안내\n1.1 사업 개요\n가. 사업명 : 한국원자력연구원 선량평가시스템...



period candidate rows: 14817
eligibility candidate rows: 21902

### 기간/기한 후보 df - 상위 10행


,pilot_doc_id,doc_id,source_file,period_type,period_value,score,score_reasons,is_form_context,line_index,raw_text,context
0,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,계약일로부터 24개월 이내,100,"project_duration_keyword, period_value, direct...",False,41,나. 사업기간: 계약일로부터 24개월 이내,가. 사업명: 고려대학교 차세대 포털·학사 정보시스템 구축 사업 나. 사업기간: 계...
1,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,계약일로부터 24개월 이내,92,"project_duration_context, period_value, direct...",False,40,가. 사업명: 고려대학교 차세대 포털·학사 정보시스템 구축 사업,1. 사업개요 가. 사업명: 고려대학교 차세대 포털·학사 정보시스템 구축 사업 나....
2,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,64,"project_duration_keyword, direct_project_perio...",False,666,"❍ 프로젝트 수행 기간 중 사업관리 조직을 구성하고, PM(Project Manag...",검토회의 등은 프로젝트 관리도구의 데이터를 근거로 수행하여야 함 ❍ 프로젝트 수행 ...
3,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,64,"project_duration_keyword, direct_project_perio...",False,728,❍ 차세대 사업 기간 중 진행되는 DBMS 버전 업그레이드 작업과 관련하여 차세대,방안을 제시하여야 함 ❍ 차세대 사업 기간 중 진행되는 DBMS 버전 업그레이드 작...
4,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,64,"project_duration_keyword, direct_project_perio...",False,6173,- 사업기간 내 보안상 문제점 발견시 즉시 대책을 수립 후 해결방안 제출,"- 사업수행조직 내 보안·개인정보 책임자, 담당자를 지정하여 운영 - 사업기간 내 ..."
5,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,64,"project_duration_keyword, direct_project_perio...",False,6181,기밀은 과업기간 및 본 계약 종료 후에도 제3자에게 누설 금지,- 우리대학의 서면에 의한 승낙 없이 본 계약에 관련하여 알게 된 업무상 기밀은 과...
6,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,64,"project_duration_keyword, direct_project_perio...",False,6493,"- 사업기간 동안 이루어질 보고 및 검토계획을 상세하게 제시(착수, 중간,","- 단계별 일정, 우선순위, 세부활동 등이 포함된 일정계획 제시 - 사업기간 동안 ..."
7,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,64,"project_duration_keyword, direct_project_perio...",False,6506,대학의 사유로 사업기간의 연장이 불가피 할 때 대학의 승인을 얻어 기간을,"- 천재지변 등 불가항력으로 사업수행에 지장이 생겼을 때와, 설계 변경 또는 대학의..."
8,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,64,"project_duration_keyword, direct_project_perio...",False,6561,- 우리대학과 협의하여 사업기간 중 관계자 워크숍 및 설명회 등 개최 가능함,시기 및 진행방법은 우리대학과 협의하여 결정함 - 우리대학과 협의하여 사업기간 중 ...
9,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,64,"project_duration_keyword, direct_project_perio...",False,6627,- 사업관리자(PM)는 본 사업기간동안 특별한 사정이 없는 한 변경될 수,- 사업관리자(PM)는 반드시 주관사업자 소속이어야 함 - 사업관리자(PM)는 본 ...



### 입찰참가자격 후보 df - 상위 10행


,pilot_doc_id,doc_id,source_file,matched_terms,score,score_reasons,is_form_context,line_index,raw_text,context
0,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,"입찰참가자격, 참가자격, 입찰참가, 공동수급",70,"eligibility_heading, eligibility_terms, condit...",False,6982,에 저촉되지 않아야 함,사자로 하는 계약에 관한 법률 시행령 제12조 및 동법 시행규칙 제14조 규정 에 ...
1,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,"입찰참가자격, 참가자격, 입찰참가, 공동수급",70,"eligibility_heading, eligibility_terms, condit...",False,6983,- 공동수급업체 참여자는 입찰참가자격을 모두 충족하여야 함,에 저촉되지 않아야 함 - 공동수급업체 참여자는 입찰참가자격을 모두 충족하여야 함 ...
2,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,"입찰참가자격, 참가자격, 입찰참가, 공동수급",70,"eligibility_heading, eligibility_terms, condit...",False,6984,- 공동수급협정서를 제출하여야 함,- 공동수급업체 참여자는 입찰참가자격을 모두 충족하여야 함 - 공동수급협정서를 제출...
3,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,"참가 자격, 소프트웨어사업자",56,"eligibility_heading, eligibility_terms, condit...",False,6969,1) 소프트웨어산업진흥법 제24조 규정에 의거 소프트웨어사업자(컴퓨터관련 서,가. 참가 자격 1) 소프트웨어산업진흥법 제24조 규정에 의거 소프트웨어사업자(컴퓨...
4,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,"참가 자격, 소프트웨어사업자",48,"eligibility_heading, eligibility_terms, short_...",False,6968,가. 참가 자격,3. 사업자 선정 방법 가. 참가 자격 1) 소프트웨어산업진흥법 제24조 규정에 의...
5,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,참가 자격,41,"eligibility_heading, eligibility_terms, short_...",False,6967,3. 사업자 선정 방법,-233- 3. 사업자 선정 방법 가. 참가 자격
6,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,"대기업, 중소기업, 소기업, 소프트웨어사업자",40,"eligibility_terms, condition_phrase, short_line",False,6914,10) 소프트웨어사업자 등록증 사본,9) 입찰보증금 (하단 다.-1) 참조) 10) 소프트웨어사업자 등록증 사본 11)...
7,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,"대기업, 중소기업, 소기업, 소프트웨어사업자",40,"eligibility_terms, condition_phrase, short_line",False,6915,"11) 기업 분류 확인 자료(중소기업, 중견기업, 대기업 등)","10) 소프트웨어사업자 등록증 사본 11) 기업 분류 확인 자료(중소기업, 중견기업..."
8,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,참가자격,29,"eligibility_heading, eligibility_terms, short_...",True,8078,"가. 제출된 모든 관련 증빙서류는 성실하게 작성 제출하며,","- 아 래 - 가. 제출된 모든 관련 증빙서류는 성실하게 작성 제출하며, 만일 허위..."
9,P001,doc_872ec563d3,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,참가자격,29,"eligibility_heading, eligibility_terms, short_...",True,8079,만일 허위 기재사항 등이 확인될 경우에는 참가자격에서 제외되어도,"가. 제출된 모든 관련 증빙서류는 성실하게 작성 제출하며, 만일 허위 기재사항 등이..."



--- chunk preview ---
chunk_id: P001_v2_toc_0001_part_001
source_file: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
chunk_type: toc
issuer: 고려대학교
project_name: 차세대 포털·학사 정보시스템 구축사업
final_notice_id: 
final_budget: 11,270,000,000원
final_bid_deadline: 
final_project_duration: 계약일로부터 24개월 이내
final_maintenance_period: 사업종료일로부터 12개월
final_warranty_period: 12개월
final_deadline_terms: 
final_bid_eligibility_terms: - 공동수급업체 참여자는 입찰참가자격을 모두 충족하여야 함 | 1) 소프트웨어산업진흥법 제24조 규정에 의거 소프트웨어사업자(컴퓨터관련 서 | 가. 참가 자격
exact_terms: ['고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', '고려대학교_차세대 포털·학사 정보시스템 구축사업', '차세대 포털·학사 정보시스템 구축사업', '고려대학교', '11,270,000,000원', '11270000000', '계약일로부터 24개월 이내', '사업종료일로부터 12개월']
amounts: []
dates: ['2024.7.01']
[문서: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf | 사업명: 차세대 포털·학사 정보시스템 구축사업 | 발주기관: 고려대학교 | 섹션: 목차 | 유형: toc]
2024. 7. 01
목 차
Ⅰ. 사업개요
1. 사업 개요
2. 사업 배경
3. 사업 범위
4. 기대효과
Ⅱ. 현황 및 문제점
1. 일반 현황
2. 정보화 현황
3. 문제점 및 개선방향
Ⅲ. 사업 추진방안
1. 추진 목표
2. 추진 일정
3. 추진 체계
Ⅳ. 제안요청 내용
1. 제안요청 개요
2. 상세 요구사항
Ⅴ. 제안 일반 사항
1. 사업관리 사항
2. 입찰 

## Sanity Checks

The cells below inspect the already-created `parsing_p2_250` artifacts.
They do not rebuild parsing outputs.


In [7]:
# 6. Sanity check: artifact existence, workbook sheets, and row counts

import json
import re
from pathlib import Path

import pandas as pd

revision_dir = PATHS["parsing_output_dir"]
excel_path = PATHS["metadata_excel"]

required_artifacts = [
    "pilot_docs_250.csv",
    "doc_parse_summary.csv",
    "parsed_blocks_v1.jsonl",
    "parsed_blocks_v2.jsonl",
    "chunks_v1.jsonl",
    "chunks_v2.jsonl",
    "parsing_summary.json",
    "parsing_summary.md",
    PATHS["metadata_excel"].name,
]

artifact_check = pd.DataFrame([
    {
        "file": name,
        "exists": (revision_dir / name).exists(),
        "size_mb": round((revision_dir / name).stat().st_size / 1024 / 1024, 2) if (revision_dir / name).exists() else None,
        "updated_at": (revision_dir / name).stat().st_mtime if (revision_dir / name).exists() else None,
    }
    for name in required_artifacts
])

summary = json.loads(PATHS["parsing_summary_json"].read_text(encoding="utf-8"))
metadata_df = pd.read_excel(excel_path, sheet_name="metadata_250")
doc_summary_df = pd.read_csv(PATHS["doc_parse_summary_csv"], encoding="utf-8-sig")
workbook = pd.ExcelFile(excel_path)

print("revision_dir:", revision_dir)
print("excel_path:", excel_path)
print("output_description:", summary.get("output_description"))
print("parsing_output_name:", summary.get("parsing_output_name"))
print("parsing_version_label:", summary.get("parsing_version_label"))
print("workbook sheets:", workbook.sheet_names)
print("metadata rows:", len(metadata_df))
print("doc summary rows:", len(doc_summary_df))
display(artifact_check)


revision_dir: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250
excel_path: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\rfp_parsing_metadata_250_p2_chunkfix_toc_clean.xlsx
output_description: P2 - chunk_id 중복 수정, toc 분리, artifact cleaner 보강
parsing_output_name: parsing_p2_250
parsing_version_label: p2_chunkfix_toc_clean
workbook sheets: ['metadata_250', 'summary', 'doc_parse_summary', 'notice_id_candidates', 'budget_candidates', 'date_candidates', 'period_candidates', 'eligibility_candidates', 'submission_candidates', 'final_submission_docs', 'final_eligibility_terms']
metadata rows: 250
doc summary rows: 250


,file,exists,size_mb,updated_at
0,pilot_docs_250.csv,True,0.13,1.778966e+09
1,doc_parse_summary.csv,True,0.63,1.778966e+09
2,parsed_blocks_v1.jsonl,True,193.67,1.778966e+09
3,parsed_blocks_v2.jsonl,True,330.40,1.778966e+09
4,chunks_v1.jsonl,True,390.71,1.778966e+09
5,chunks_v2.jsonl,True,510.61,1.778966e+09
6,parsing_summary.json,True,0.00,1.778966e+09
7,parsing_summary.md,True,0.00,1.778966e+09
8,rfp_parsing_metadata_250_p2_chunkfix_toc_clean...,True,10.71,1.778966e+09


In [8]:
# 7. Sanity check: key metadata coverage and missing examples

def present(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.strip()

coverage_columns = [
    "공고 번호",
    "사업 금액",
    "사업금액_원",
    "공개 일자",
    "입찰 참여 시작일",
    "입찰 참여 마감일",
    "사업기간",
    "무상유지보수기간",
    "하자담보책임기간",
    "기한/기간 기타",
    "입찰참가자격",
    "제출서류",
]

coverage_rows = []
for column in coverage_columns:
    values = present(metadata_df[column]) if column in metadata_df.columns else pd.Series([""] * len(metadata_df))
    coverage_rows.append({
        "column": column,
        "filled_docs": int((values != "").sum()),
        "missing_docs": int((values == "").sum()),
        "filled_rate": round(float((values != "").mean()), 3),
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

print("budget status counts:")
display(metadata_df["사업금액_상태"].fillna("missing").value_counts(dropna=False).rename_axis("status").reset_index(name="docs"))

for label, column in [
    ("사업금액 결측 예시", "사업 금액"),
    ("공개일자 결측 예시", "공개 일자"),
    ("입찰시작일 결측 예시", "입찰 참여 시작일"),
    ("입찰마감일 결측 예시", "입찰 참여 마감일"),
    ("공고번호 결측 예시", "공고 번호"),
]:
    missing = metadata_df[present(metadata_df[column]) == ""]
    print("\n" + label + f": {len(missing)} docs")
    display(missing[["파일명", "사업명", "발주 기관", "원문텍스트_20000자"]].head(5))


,column,filled_docs,missing_docs,filled_rate
0,공고 번호,2,248,0.008
1,사업 금액,238,12,0.952
2,사업금액_원,238,12,0.952
3,공개 일자,46,204,0.184
4,입찰 참여 시작일,7,243,0.028
5,입찰 참여 마감일,40,210,0.160
6,사업기간,245,5,0.980
7,무상유지보수기간,53,197,0.212
8,하자담보책임기간,203,47,0.812
9,기한/기간 기타,151,99,0.604


budget status counts:


,status,docs
0,extracted,238
1,missing,10
2,candidate_only,2



사업금액 결측 예시: 12 docs


,파일명,사업명,발주 기관,원문텍스트_20000자
2,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp,우즈벡-키르기즈스탄 기후변화대응 스,사단법인아시아물위원회사무국,목 차\nⅠ. 제안 요청내용 01\n1. 용역개요 02\n2. 기본방향 03\n3....
8,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,[재공고]차세대 통합정보시스템(ERP) 구축,한국가스공사,목 차\n\n과 업 지 시 서\n- 차세대 통합정보시스템(ERP) 구축 -\n202...
32,사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp,실손보험 청구 전산화 시스템 구축 사업,사단법인 보험개발원,목 차\n1. 개요 2\n2. 추진배경 및 필요성 2\n3. 실손보험 청구 전산화 ...
33,을지대학교_을지대학교 비교과시스템 개발.hwp,을지대학교 비교과시스템 개발,을지대학교,목 차\nⅠ. 사업 개요 4\n1. 사업개요 4\n2. 사업범위 4\nⅡ. 현황 5...
35,국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp,기록관리시스템 통합 활용 및 보안 환경 구축,국방과학연구소,목 차\nⅠ. 사업 개요 4\n1. 사업 정보 4\n2. 추진배경 및 필요성 4\n...


...

,파일명,사업명,발주 기관,원문텍스트_20000자
0,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,차세대 포털·학사 정보시스템 구축사업,고려대학교,2024. 7. 01\n목 차\nⅠ. 사업개요\n1. 사업 개요\n2. 사업 배경\...
1,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원...,2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시,(사)부산국제영화제,목차\n목 차\n\n제 안 요 청 서\nBIFF&ACFM 온라인서비스 재개발 및\n...
2,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp,우즈벡-키르기즈스탄 기후변화대응 스,사단법인아시아물위원회사무국,목 차\nⅠ. 제안 요청내용 01\n1. 용역개요 02\n2. 기본방향 03\n3....
3,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp,의료기기산업 종합정보시스템(정보관리기관) 기능,한국보건산업진흥원,목 차\n1. 사업개요 4\n2. 추진배경 및 필요성 4\n3. 과업내용 5\n4....
4,한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp,한국원자력연구원 선량평가시스템 고도화,한국원자력연구원,1\n사업 안내\n1.1 사업 개요\n가. 사업명 : 한국원자력연구원 선량평가시스템...



입찰마감일 결측 예시: 210 docs


,파일명,사업명,발주 기관,원문텍스트_20000자
0,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,차세대 포털·학사 정보시스템 구축사업,고려대학교,2024. 7. 01\n목 차\nⅠ. 사업개요\n1. 사업 개요\n2. 사업 배경\...
2,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp,우즈벡-키르기즈스탄 기후변화대응 스,사단법인아시아물위원회사무국,목 차\nⅠ. 제안 요청내용 01\n1. 용역개요 02\n2. 기본방향 03\n3....
3,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp,의료기기산업 종합정보시스템(정보관리기관) 기능,한국보건산업진흥원,목 차\n1. 사업개요 4\n2. 추진배경 및 필요성 4\n3. 과업내용 5\n4....
4,한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp,한국원자력연구원 선량평가시스템 고도화,한국원자력연구원,1\n사업 안내\n1.1 사업 개요\n가. 사업명 : 한국원자력연구원 선량평가시스템...
5,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,예약발매시스템 개량 ISMP 용역,한국철도공사 (용역),1. 사업 개요 1\n2. 현황 및 문제점 2\n3. 추진방안 5\n4. 추진체계 ...



공고번호 결측 예시: 248 docs


,파일명,사업명,발주 기관,원문텍스트_20000자
0,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,차세대 포털·학사 정보시스템 구축사업,고려대학교,2024. 7. 01\n목 차\nⅠ. 사업개요\n1. 사업 개요\n2. 사업 배경\...
1,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원...,2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시,(사)부산국제영화제,목차\n목 차\n\n제 안 요 청 서\nBIFF&ACFM 온라인서비스 재개발 및\n...
2,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp,우즈벡-키르기즈스탄 기후변화대응 스,사단법인아시아물위원회사무국,목 차\nⅠ. 제안 요청내용 01\n1. 용역개요 02\n2. 기본방향 03\n3....
3,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp,의료기기산업 종합정보시스템(정보관리기관) 기능,한국보건산업진흥원,목 차\n1. 사업개요 4\n2. 추진배경 및 필요성 4\n3. 과업내용 5\n4....
4,한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp,한국원자력연구원 선량평가시스템 고도화,한국원자력연구원,1\n사업 안내\n1.1 사업 개요\n가. 사업명 : 한국원자력연구원 선량평가시스템...


In [9]:
# 8. Sanity check: P2-specific schema and anti-regression checks

required_sheets = {
    "metadata_250",
    "summary",
    "doc_parse_summary",
    "notice_id_candidates",
    "budget_candidates",
    "date_candidates",
    "period_candidates",
    "eligibility_candidates",
    "submission_candidates",
    "final_submission_docs",
    "final_eligibility_terms",
}
required_columns = {
    "\uc0ac\uc5c5\uae30\uac04",
    "\uc0ac\uc5c5\uae30\uac04_\uadfc\uac70",
    "\ubb34\uc0c1\uc720\uc9c0\ubcf4\uc218\uae30\uac04",
    "\ubb34\uc0c1\uc720\uc9c0\ubcf4\uc218\uae30\uac04_\uadfc\uac70",
    "\ud558\uc790\ub2f4\ubcf4\ucc45\uc784\uae30\uac04",
    "\ud558\uc790\ub2f4\ubcf4\ucc45\uc784\uae30\uac04_\uadfc\uac70",
    "\uae30\ud55c/\uae30\uac04 \uae30\ud0c0",
    "\uae30\ud55c/\uae30\uac04 \uae30\ud0c0_\uadfc\uac70",
    "\uc785\ucc30\ucc38\uac00\uc790\uaca9",
    "\uc785\ucc30\ucc38\uac00\uc790\uaca9_\uadfc\uac70",
    "\uc6d0\ubb38\ud14d\uc2a4\ud2b8_20000\uc790",
}

unknown_notice_header_count = 0
unknown_notice_text = "\uacf5\uace0\ubc88\ud638: unknown"
with PATHS["chunks_v2_jsonl"].open("r", encoding="utf-8") as f:
    for line in f:
        if unknown_notice_text in line:
            unknown_notice_header_count += 1

notice_col_left = "\uacf5\uace0"
notice_col_right = "\ubc88\ud638"
notice_column_count = sum(1 for col in metadata_df.columns if notice_col_left in str(col) and notice_col_right in str(col))
schema_check = {
    "is_p2_output": summary.get("parsing_output_name") == "parsing_p2_250",
    "is_p2_label": summary.get("parsing_version_label") == "p2_chunkfix_toc_clean",
    "is_p2_description": "P2" in str(summary.get("output_description", "")),
    "is_not_v3_label": "v3" not in str(summary.get("parsing_version_label", "")).lower(),
    "required_sheets_present": required_sheets.issubset(set(workbook.sheet_names)),
    "required_columns_present": required_columns.issubset(set(metadata_df.columns)),
    "notice_column_count_is_1": notice_column_count == 1,
    "unknown_notice_header_count_is_0": unknown_notice_header_count == 0,
    "pilot_docs_250": len(metadata_df) == 250,
    "parse_success_250": int(summary.get("parse_success_docs", 0)) == 250,
}

display(pd.DataFrame([{"check": key, "passed": value} for key, value in schema_check.items()]))
print("notice_column_count:", notice_column_count)
print("unknown_notice_header_count:", unknown_notice_header_count)
assert all(schema_check.values()), "One or more P2 sanity checks failed."


,check,passed
0,is_p2_output,True
1,is_p2_label,True
2,is_p2_description,True
3,is_not_v3_label,True
4,required_sheets_present,True
5,required_columns_present,True
6,notice_column_count_is_1,True
7,unknown_notice_header_count_is_0,True
8,pilot_docs_250,True
9,parse_success_250,True


notice_column_count: 1
unknown_notice_header_count: 0


In [10]:
# 9. Sanity check: sample documents for revised budget/period/eligibility extraction

period_candidates_df = pd.read_excel(excel_path, sheet_name="period_candidates")
eligibility_candidates_df = pd.read_excel(excel_path, sheet_name="eligibility_candidates")
budget_candidates_df = pd.read_excel(excel_path, sheet_name="budget_candidates")

sample_name_parts = [
    "고려대학교",
    "한국가스공사",
    "한국수자원공사_수도사업장 통합 사고분석솔루션",
]

sample_cols = [
    "파일명",
    "사업 금액",
    "사업금액_원",
    "사업기간",
    "사업기간_근거",
    "무상유지보수기간",
    "무상유지보수기간_근거",
    "하자담보책임기간",
    "하자담보책임기간_근거",
    "기한/기간 기타",
    "기한/기간 기타_근거",
    "입찰참가자격",
]

for name_part in sample_name_parts:
    sample = metadata_df[metadata_df["파일명"].astype(str).str.contains(name_part, regex=False, na=False)]
    print("\n=== sample:", name_part, "===")
    display(sample[sample_cols].head(3))
    if sample.empty:
        continue
    source_file = sample.iloc[0]["파일명"]
    print("period candidates:")
    display(
        period_candidates_df[period_candidates_df["source_file"].astype(str) == str(source_file)]
        [["period_type", "period_value", "score", "is_form_context", "raw_text"]]
        .head(10)
    )
    print("budget candidates:")
    display(
        budget_candidates_df[budget_candidates_df["source_file"].astype(str) == str(source_file)]
        [["raw_amount", "amount_krw", "score", "context"]]
        .head(8)
    )
    print("eligibility candidates:")
    display(
        eligibility_candidates_df[eligibility_candidates_df["source_file"].astype(str) == str(source_file)]
        [["matched_terms", "score", "is_form_context", "raw_text"]]
        .head(8)
    )



=== sample: 고려대학교 ===


,파일명,사업 금액,사업금액_원,사업기간,사업기간_근거,무상유지보수기간,무상유지보수기간_근거,하자담보책임기간,하자담보책임기간_근거,기한/기간 기타,기한/기간 기타_근거,입찰참가자격
0,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,"11,270,000,000원",1.127000e+10,계약일로부터 24개월 이내,가. 사업명: 고려대학교 차세대 포털·학사 정보시스템 구축 사업 나. 사업기간: 계...,사업종료일로부터 12개월,나. 사업기간: 계약일로부터 24개월 이내 다. 무상유지보수기간 : 사업종료일로부터...,12개월,"보장하여야 하며, 우리 학교에서 운영 중인 각종 S/W와 연동이 되어야 함 ❍ 무상...",NaN,NaN,- 공동수급업체 참여자는 입찰참가자격을 모두 충족하여야 함 | 1) 소프트웨어산업진...


period candidates:


,period_type,period_value,score,is_form_context,raw_text
0,project_duration,계약일로부터 24개월 이내,100,False,나. 사업기간: 계약일로부터 24개월 이내
1,project_duration,계약일로부터 24개월 이내,92,False,가. 사업명: 고려대학교 차세대 포털·학사 정보시스템 구축 사업
2,project_duration,NaN,64,False,"❍ 프로젝트 수행 기간 중 사업관리 조직을 구성하고, PM(Project Manag..."
3,project_duration,NaN,64,False,❍ 차세대 사업 기간 중 진행되는 DBMS 버전 업그레이드 작업과 관련하여 차세대
4,project_duration,NaN,64,False,- 사업기간 내 보안상 문제점 발견시 즉시 대책을 수립 후 해결방안 제출
5,project_duration,NaN,64,False,기밀은 과업기간 및 본 계약 종료 후에도 제3자에게 누설 금지
6,project_duration,NaN,64,False,"- 사업기간 동안 이루어질 보고 및 검토계획을 상세하게 제시(착수, 중간,"
7,project_duration,NaN,64,False,대학의 사유로 사업기간의 연장이 불가피 할 때 대학의 승인을 얻어 기간을
8,project_duration,NaN,64,False,- 우리대학과 협의하여 사업기간 중 관계자 워크숍 및 설명회 등 개최 가능함
9,project_duration,NaN,64,False,- 사업관리자(PM)는 본 사업기간동안 특별한 사정이 없는 한 변경될 수


budget candidates:


...

,period_type,period_value,score,is_form_context,raw_text
704,project_duration,4 개월,100,False,사업기간
705,project_duration,착수일로부터 120일간,92,False,"나. 사업금액 : 일금 일억구천오백삼만원정(￦195,030,000, VAT포함)"
706,project_duration,4 개월 「소프트웨어사업 계약 및 관리감독에 관한 지침」제10조제3항에 따른 소프트...,92,False,4 개월
707,project_duration,계약일로부터 4개월,92,False,(또는 개발기간)
708,project_duration,착수일로부터 120일간,82,True,다. 사업기간 : 착수일로부터 120일간
709,project_duration,착수일로부터 120일간,82,True,※ [붙임 1] 소프트웨어 개발사업의 적정 사업기간 종합 산정서 참조
710,project_duration,4개월,82,False,"수도사업장 내 통합 사고분석을 위한 솔루션의 개발 및 현장 적용, 시운전의 과업을 ..."
711,project_duration,4 개월,82,False,「소프트웨어사업 계약 및 관리감독에 관한 지침」제10조제3항에 따른 소프트웨어 개발...
712,project_duration,4개월,74,False,⑤ 종합의견
713,project_duration,4개월,74,False,적정


budget candidates:


,raw_amount,amount_krw,score,context
48,"￦195,030,000",195030000,25,준 및 작성양식 1. 제안공모 개요 1.1 일반사항 1.1.1 사업개요 가. 사 업...
49,"195,030,000",195030000,25,및 작성양식 1. 제안공모 개요 1.1 일반사항 1.1.1 사업개요 가. 사 업 명...


eligibility candidates:


,matched_terms,score,is_form_context,raw_text
1010,"입찰참가자격, 참가자격, 입찰참가, 공동수급",70,False,나. 공동도급으로 참여할 경우 공동수급체 대표는 출자비율이 가장 높은 업체이어야 함
1011,"입찰참가자격, 참가자격, 입찰참가, 공동수급",70,False,다. 공동수급체 대표사 및 구성원은 “입찰참가자격”의 모든 자격을 갖추어야 함
1012,"입찰참가자격, 참가자격, 입찰참가, 공동수급",70,False,라. 공동수급체 구성원이 다른 공동수급체를 구성하여 중복 입찰참가는 불가함
1013,"입찰참가자격, 참가자격, 입찰참가",63,False,◦ 계약자가 반출된 SW산출물을 제3자에게 제공하려는 경우 반드시 발주기관으로부터 ...
1014,"입찰참가자격, 참가자격, 입찰참가",63,False,◦ 발주기관은 공급자가 제공받은 SW산출물을 무단으로 유출하거나 누출되는 경우 및 ...
1015,"입찰참가자격, 참가자격, 입찰참가",63,False,고유번호
1016,"참가자격, 부정당업자",56,False,7) 품질 요구사항
1017,"입찰참가자격, 참가자격, 입찰참가",55,False,㉱ 법인등기부등본 1부(법인인 경우에 한함)


In [11]:
# 10. Sanity check: suspicious final values and candidate/final separation

period_final_columns = [
    "사업기간",
    "무상유지보수기간",
    "하자담보책임기간",
    "기한/기간 기타",
]

suspicious_patterns = [
    r"\b0+\d+\s*(일|개월|년)",
    r"\b\d{1,2}\s*일\b",
    r"제\s*\d+\s*(조|항|호)",
]

suspicious_rows = []
for column in period_final_columns:
    values = present(metadata_df[column])
    for pattern in suspicious_patterns:
        hits = metadata_df[values.str.contains(pattern, regex=True, na=False)]
        for _, row in hits.head(20).iterrows():
            suspicious_rows.append({
                "column": column,
                "pattern": pattern,
                "파일명": row.get("파일명", ""),
                "value": row.get(column, ""),
            })

suspicious_df = pd.DataFrame(suspicious_rows)
print("suspicious final period/deadline values:", len(suspicious_df))
display(suspicious_df.head(30))

form_context_candidates = period_candidates_df[period_candidates_df["is_form_context"].astype(str).str.lower().isin(["true", "1"])]
print("period candidates kept only as form/template context:", len(form_context_candidates))
display(form_context_candidates[["source_file", "period_type", "period_value", "score", "raw_text"]].head(10))

external_deadline_candidates = period_candidates_df[
    period_candidates_df["period_value"].fillna("").astype(str).str.contains("공고|참조|따름", regex=True)
]
print("external-reference deadline candidates:", len(external_deadline_candidates))
display(external_deadline_candidates[["source_file", "period_type", "period_value", "score", "raw_text"]].head(10))


suspicious final period/deadline values: 0


C:\Users\yoosy\AppData\Local\Temp\ipykernel_28388\3408062993.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = metadata_df[values.str.contains(pattern, regex=True, na=False)]
C:\Users\yoosy\AppData\Local\Temp\ipykernel_28388\3408062993.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = metadata_df[values.str.contains(pattern, regex=True, na=False)]
C:\Users\yoosy\AppData\Local\Temp\ipykernel_28388\3408062993.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = metadata_df[values.str.contains(pattern, regex=True, na=False)]
C:\Users\yoosy\AppData\Local\Temp\ipykernel_28388\3408062993.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the gr

""


period candidates kept only as form/template context: 3310


,source_file,period_type,period_value,score,raw_text
61,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,46,사 업 명 사 업 기 간 계 약 금 액 발 주 처 비 고
62,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,46,사업명 사업기간 개월
63,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,46,계약금액(C) 원 사업기간 년 월 일 ∼ 년 월 일
64,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,46,계약기간 하도급예정액(A) 계약금액 대비
76,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,38,주요사업 실적
77,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,38,"※ 연도순으로 기재하며, 제안과제와 유사하거나 동일한 업무영역이나 사업형태에 관한"
78,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,38,계약
79,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,38,소프트웨어사업 하도급 계획서(입찰 시)
80,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,38,금액(B) 원
81,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,project_duration,NaN,38,사업명


external-reference deadline candidates: 556


,source_file,period_type,period_value,score,raw_text
178,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp,deadline_term,공고문 참조,48,○ 제출일시 및 장소 : 입찰공고문 참조
194,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp,deadline_term,공고문 참조,44,○ e발주시스템 온라인 제출
195,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp,deadline_term,공고문 참조,44,5
313,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,deadline_term,공고문 참조,48,◦ 제안서 제출방법 : 입찰공고문 참조
314,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,deadline_term,공고문 참조,48,◦ 제안서 접수기간 : 입찰공고문 참조
319,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,deadline_term,공고문 참조,44,2. 제안서 제출
320,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,deadline_term,공고문 참조,43,3. 입찰보증금 : 입찰공고문 참조
330,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,deadline_term,공고문 참조,39,4. 첨부할 증빙서류
452,케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp,deadline_term,공고문 참조,48,제출처 및 기한 : 입찰공고문 참조
453,케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp,deadline_term,공고문 참조,48,제안서 제출 기한 및 방법: 입찰공고문 참조


In [12]:
# 11. Sanity check: compact final report for team sharing

report_rows = [
    ("전체 문서 수", len(metadata_df)),
    ("파싱 성공 문서 수", summary.get("parse_success_docs")),
    ("공고번호 final 존재", int((present(metadata_df["공고 번호"]) != "").sum())),
    ("사업금액 final 존재", int((present(metadata_df["사업 금액"]) != "").sum())),
    ("공개일자 final 존재", int((present(metadata_df["공개 일자"]) != "").sum())),
    ("입찰시작일 final 존재", int((present(metadata_df["입찰 참여 시작일"]) != "").sum())),
    ("입찰마감일 final 존재", int((present(metadata_df["입찰 참여 마감일"]) != "").sum())),
    ("사업기간 final 존재", int((present(metadata_df["사업기간"]) != "").sum())),
    ("무상유지보수기간 final 존재", int((present(metadata_df["무상유지보수기간"]) != "").sum())),
    ("하자담보책임기간 final 존재", int((present(metadata_df["하자담보책임기간"]) != "").sum())),
    ("기한/기간 기타 final 존재", int((present(metadata_df["기한/기간 기타"]) != "").sum())),
    ("입찰참가자격 final 존재", int((present(metadata_df["입찰참가자격"]) != "").sum())),
    ("제출서류 final 존재", int((present(metadata_df["제출서류"]) != "").sum())),
    ("period candidate rows", len(period_candidates_df)),
    ("eligibility candidate rows", len(eligibility_candidates_df)),
    ("unknown 공고번호 header", unknown_notice_header_count),
]

final_report_df = pd.DataFrame(report_rows, columns=["metric", "value"])
display(final_report_df)

print("해석 메모:")
print("- 사업금액/일자/기간은 원문에서 확실하게 찾은 값만 final로 올립니다.")
print("- 공고문 참조/입찰 공고에 따름은 날짜로 만들지 않고 기한/기간 기타 또는 candidate 근거로 둡니다.")
print("- 가격제안서/하도급계획서/별지/서식 안의 작성 공란은 final 승격을 억제합니다.")
print("- candidate sheet는 넓게 잡은 검수 후보이고, metadata_250의 final 컬럼은 더 보수적으로 선별한 값입니다.")


,metric,value
0,전체 문서 수,250
1,파싱 성공 문서 수,250
2,공고번호 final 존재,2
3,사업금액 final 존재,238
4,공개일자 final 존재,46
5,입찰시작일 final 존재,7
6,입찰마감일 final 존재,40
7,사업기간 final 존재,245
8,무상유지보수기간 final 존재,53
9,하자담보책임기간 final 존재,203


해석 메모:
- 사업금액/일자/기간은 원문에서 확실하게 찾은 값만 final로 올립니다.
- 공고문 참조/입찰 공고에 따름은 날짜로 만들지 않고 기한/기간 기타 또는 candidate 근거로 둡니다.
- 가격제안서/하도급계획서/별지/서식 안의 작성 공란은 final 승격을 억제합니다.
- candidate sheet는 넓게 잡은 검수 후보이고, metadata_250의 final 컬럼은 더 보수적으로 선별한 값입니다.
